run once, before the app

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier, XGBRegressor

In [3]:
DATA_PATH = Path("/content/EMI_dataset.csv")
OUT_PATH  = Path("emi_artifacts.joblib")

In [4]:
df = pd.read_csv(DATA_PATH)
df = df.dropna().drop_duplicates()
df = df.sample(n=50_000, random_state=42).reset_index(drop=True)

In [5]:
cat_cols = ["gender", "marital_status", "education", "employment_type",
            "company_type", "house_type", "emi_scenario"]
category_values = {c: sorted(df[c].dropna().unique().tolist()) for c in cat_cols}

In [6]:
df["total_expenses"] = (df["monthly_rent"] + df["school_fees"] + df["college_fees"]
                        + df["travel_expenses"] + df["groceries_utilities"]
                        + df["other_monthly_expenses"] + df["current_emi_amount"])
df["disposable_income"]       = df["monthly_salary"] - df["total_expenses"]
df["debt_to_income"]          = df["current_emi_amount"] / df["monthly_salary"]
df["expense_to_income"]       = df["total_expenses"] / df["monthly_salary"]
df["savings_ratio"]           = df["bank_balance"] / (df["monthly_salary"] * 12)
df["requested_emi_estimate"]  = df["requested_amount"] / df["requested_tenure"]
df["requested_emi_to_income"] = df["requested_emi_estimate"] / df["monthly_salary"]

In [7]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [8]:
num_cols = ["age", "monthly_salary", "years_of_employment", "monthly_rent",
            "family_size", "dependents", "school_fees", "college_fees",
            "travel_expenses", "groceries_utilities", "other_monthly_expenses",
            "existing_loans", "current_emi_amount", "credit_score",
            "bank_balance", "emergency_fund", "requested_amount", "requested_tenure",
            "total_expenses", "disposable_income", "debt_to_income",
            "expense_to_income", "savings_ratio", "requested_emi_estimate",
            "requested_emi_to_income"]

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [9]:
X     = df.drop(columns=["emi_eligibility", "max_monthly_emi"])
y_clf = df["emi_eligibility"]
y_reg = df["max_monthly_emi"]

feature_columns = list(X.columns)   # <-- critical for inference alignment

In [10]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

le = LabelEncoder()
y_tr_enc = le.fit_transform(y_tr)

xgb_clf = XGBClassifier(random_state=42, eval_metric="mlogloss")
xgb_clf.fit(X_tr, y_tr_enc)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [11]:
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

xgb_reg = XGBRegressor(random_state=42)
xgb_reg.fit(X_tr_r, y_tr_r)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [12]:
artifacts = {
    "classifier":       xgb_clf,
    "regressor":        xgb_reg,
    "scaler":           scaler,
    "label_encoder":    le,
    "feature_columns":  feature_columns,
    "num_cols":         num_cols,
    "cat_cols":         cat_cols,
    "category_values":  category_values,
}
joblib.dump(artifacts, OUT_PATH)

['emi_artifacts.joblib']